# Training generazionale — 5 iter × 100 partite

**Obiettivo**: vedere se la rete migliora di iterazione in iterazione. Misuriamo "rete_N batte rete_N-1" (eval inter-rete), che è il segnale forte. L'eval vs random è solo sanity check finale.

**Parametri concordati**:
- 5 iterazioni × 100 partite self-play (n_sim=10, max_dec=1500, 6 worker)
- Buffer cumulativo, finestra 3 iter
- Training 1500 step, batch 128, lr 1e-3
- Niente filtro accettazione (rete sempre sostituita)
- Eval N vs N-1: 40 partite (20 nuova=BLU + 20 nuova=ROSSO), temp=0.3
- Eval finale vs random: 40 partite (sanity, post-iter 5), temp=0.3
- Checkpoint completo dopo ogni iter su Drive
- Cella resume separata per riprendere da iter K se Colab si scollega

**Tempo stimato**: ~9 ore totali. Cella resume permette di spezzare in più sessioni Colab.

**Note**:
- Dirichlet noise alla root MCTS è la prossima PR (PR4) — risolve il problema 'concentrazione su una sola azione' osservato nell'iterazione di prova. Per ora il self-play usa temperature=1.0 nei primi 30 step, che mitiga il problema in training.
- L'eval vs random è poco informativa per reti deboli (rete random + MCTS argmax = strategia degenerata). È stato spiegato dal test diagnostico: BLU concentra tutti i rinforzi su un solo territorio.

## 0. Setup

In [ ]:
import os, sys
REPO_DIR = '/content/risiko-rl'
BRANCH = 'main'  # IMPORTANTE: PR3 con match function deve essere mergiato

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EdoMusu1991/risiko-rl.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git rev-parse --short HEAD

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q gymnasium 'torch>=2.0'

In [ ]:
# Pre-flight: tutte le funzioni di PR3+match devono essere accessibili
try:
    from alphazero.selfplay import (
        gioca_n_partite_parallele,
        gioca_n_partite_vs_random_parallele,
        gioca_n_partite_match_parallele,
    )
    print('OK: PR3+match in main')
except ImportError as e:
    raise RuntimeError(
        'gioca_n_partite_match_parallele non importabile. Mergia la fix di match in main prima di lanciare.'
    ) from e

import psutil, torch
print(f'CPU logici: {os.cpu_count()}, RAM: {psutil.virtual_memory().total/1e9:.0f} GB')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## 1. Mount Drive + run setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import datetime
# Per riprendere da una run precedente, sostituisci con il timestamp esistente.
# Per partire da zero, lascia 'new'.
RUN_TIMESTAMP = 'new'  # oppure '20260506_0900' (formato YYYYMMDD_HHMM)

if RUN_TIMESTAMP == 'new':
    RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M')

RUN_DIR = f'/content/drive/MyDrive/risiko-rl/gen_{RUN_TIMESTAMP}'
os.makedirs(RUN_DIR, exist_ok=True)
print(f'Run dir: {RUN_DIR}')
print(f'(Per riprendere questa run, RUN_TIMESTAMP={RUN_TIMESTAMP!r})')

## 2. Knobs e setup base

Modifica qui se vuoi parametri diversi. `START_ITER=0` significa partire da zero. Per riprendere da un checkpoint, vedi cella *Resume* in fondo.

In [ ]:
# === Generazionale ===
N_ITERATIONS = 5
START_ITER = 0  # 0 = parti da zero. Per resume: vedi cella in fondo.

# === Self-play per iterazione ===
N_GAMES = 100
N_SIM = 10
MAX_DEC = 1500
N_WORKER = 6
BUFFER_WINDOW = 3  # quante iter di sample tenere nel buffer

# === Training ===
BATCH_SIZE = 128
N_TRAIN_STEPS = 1500
LR = 1e-3

# === Eval inter-rete (rete_N vs rete_N-1) ===
EVAL_GAMES_PER_COLOR = 20  # 20 nuova=BLU + 20 nuova=ROSSO = 40 totali
EVAL_N_SIM = 10
EVAL_MAX_DEC = 1500
EVAL_TEMP = 0.3

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training device: {DEVICE}')
print(f'Iterazioni: da {START_ITER} a {N_ITERATIONS-1}')

## 3. Funzioni helper

In [ ]:
import time
import json
import math
import pickle
import numpy as np

from alphazero.network import RisikoNet
from alphazero.training.replay_buffer import ReplayBuffer
from alphazero.training.trainer import Trainer


def salva_iter_checkpoint(iter_idx, net, partite_per_iter_buffer, history_log, run_dir):
    """Salva tutto quanto serve per riprendere dall'iterazione successiva."""
    iter_dir = f'{run_dir}/iter_{iter_idx:02d}'
    os.makedirs(iter_dir, exist_ok=True)
    # Rete
    torch.save(net.state_dict(), f'{iter_dir}/net.pt')
    # Sample dell'iter (per ricostruzione del buffer)
    with open(f'{iter_dir}/iter_samples.pkl', 'wb') as f:
        pickle.dump(partite_per_iter_buffer, f)
    # Log cumulativo
    with open(f'{run_dir}/history.json', 'w') as f:
        json.dump(history_log, f, indent=2)
    print(f'  ✓ Checkpoint iter {iter_idx} salvato in {iter_dir}')


def carica_iter_checkpoint(iter_idx, run_dir):
    """Carica rete e sample dell'iter specificata."""
    iter_dir = f'{run_dir}/iter_{iter_idx:02d}'
    if not os.path.exists(iter_dir):
        return None
    net = RisikoNet()
    net.load_state_dict(torch.load(f'{iter_dir}/net.pt', map_location='cpu', weights_only=True))
    net.eval()
    samples = None
    sp_path = f'{iter_dir}/iter_samples.pkl'
    if os.path.exists(sp_path):
        with open(sp_path, 'rb') as f:
            samples = pickle.load(f)
    return net, samples


def wilson_ci(wins, n):
    """Wilson score 95% CI per win rate. Stima onesta su n piccolo."""
    if n == 0: return (0, 0)
    p = wins / n
    z = 1.96
    denom = 1 + z**2/n
    centro = (p + z**2/(2*n)) / denom
    raggio = z * math.sqrt((p*(1-p) + z**2/(4*n))/n) / denom
    return (max(0, centro-raggio), min(1, centro+raggio))


def eval_inter_rete(net_new, net_old, n_per_color, n_worker, base_seed,
                    n_sim, max_dec, temp):
    """
    Misura win rate di net_new vs net_old.
    Round-robin: n_per_color partite con net_new=BLU + n_per_color con net_new=ROSSO.
    Ritorna (win_rate, ci, dettaglio_stats).
    """
    # Half 1: net_new gioca BLU
    stats_blu = gioca_n_partite_match_parallele(
        net_blu=net_new, net_rosso=net_old,
        n_partite=n_per_color, n_worker=n_worker, base_seed=base_seed,
        n_simulations=n_sim, max_decisioni=max_dec, temperature=temp,
    )
    wins_new_as_blu = sum(1 for s in stats_blu if s['vincitore'] == 'BLU')

    # Half 2: net_new gioca ROSSO
    stats_rosso = gioca_n_partite_match_parallele(
        net_blu=net_old, net_rosso=net_new,
        n_partite=n_per_color, n_worker=n_worker, base_seed=base_seed + 5000,
        n_simulations=n_sim, max_decisioni=max_dec, temperature=temp,
    )
    wins_new_as_rosso = sum(1 for s in stats_rosso if s['vincitore'] == 'ROSSO')

    total_wins = wins_new_as_blu + wins_new_as_rosso
    total_games = 2 * n_per_color
    wr = total_wins / total_games
    ci = wilson_ci(total_wins, total_games)

    detail = {
        'wins_new_as_blu': wins_new_as_blu,
        'wins_new_as_rosso': wins_new_as_rosso,
        'total_wins': total_wins,
        'total_games': total_games,
        'win_rate': wr,
        'ci_lo': ci[0], 'ci_hi': ci[1],
    }
    return wr, ci, detail

## 4. Loop generazionale

Per ogni iterazione k da START_ITER a N_ITERATIONS-1:

1. Self-play con la rete corrente → 100 partite, ~90 min
2. Aggiungi al buffer cumulativo (sliding window)
3. Training 1500 step → ~2 min
4. Eval rete_k vs rete_k-1 (se k > 0) → ~10 min
5. Salva checkpoint su Drive

Tempo per iterazione: ~100 min. Totale 5 iter ≈ 8.5 ore.

In [ ]:
# Carica history se esiste (riprende dopo crash/disconnect)
history_path = f'{RUN_DIR}/history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history_log = json.load(f)
    print(f'History caricata: {len(history_log["iterations"])} iter precedenti')
else:
    history_log = {
        'config': {
            'N_ITERATIONS': N_ITERATIONS, 'N_GAMES': N_GAMES, 'N_SIM': N_SIM,
            'MAX_DEC': MAX_DEC, 'BUFFER_WINDOW': BUFFER_WINDOW,
            'BATCH_SIZE': BATCH_SIZE, 'N_TRAIN_STEPS': N_TRAIN_STEPS, 'LR': LR,
            'EVAL_GAMES_PER_COLOR': EVAL_GAMES_PER_COLOR, 'EVAL_TEMP': EVAL_TEMP,
        },
        'iterations': [],
    }

# Carica rete iniziale: se START_ITER=0, parti da zero. Altrimenti riprendi.
if START_ITER == 0:
    torch.manual_seed(0)
    net = RisikoNet().to(DEVICE)
    net.eval()
    # Salva rete iter -1 (= iniziale) per eval iter 0 vs iniziale
    iter_neg1_dir = f'{RUN_DIR}/iter_-1'
    os.makedirs(iter_neg1_dir, exist_ok=True)
    torch.save(net.state_dict(), f'{iter_neg1_dir}/net.pt')
    print(f'Inizio da rete random (iter -1 salvata come baseline)')
else:
    res = carica_iter_checkpoint(START_ITER - 1, RUN_DIR)
    if res is None:
        raise RuntimeError(f'Non posso riprendere: iter {START_ITER-1} non trovata')
    net, _ = res
    net.to(DEVICE)
    print(f'Ripreso da iter {START_ITER-1}')

# Costruisci buffer cumulativo: prendi le ultime BUFFER_WINDOW iter precedenti
buffer = ReplayBuffer(max_size=500_000, seed=0)
for k in range(max(0, START_ITER - BUFFER_WINDOW), START_ITER):
    res = carica_iter_checkpoint(k, RUN_DIR)
    if res is not None and res[1] is not None:
        for partita_samples in res[1]:
            buffer.add_partita(partita_samples)
        print(f'  Caricato iter {k} nel buffer ({len(buffer)} sample)')

print(f'Buffer iniziale: {len(buffer)} sample')

In [ ]:
# === LOOP GENERAZIONALE ===
# Salva sample per iter (lista di liste = una lista per partita)
# che servono per ricostruire il buffer cumulativo nelle iter future
per_iter_samples_storage = {}  # iter_idx -> list of partite_samples

# Pre-carica per_iter_samples delle iter precedenti (per sliding window futuro)
for k in range(max(0, START_ITER - BUFFER_WINDOW), START_ITER):
    res = carica_iter_checkpoint(k, RUN_DIR)
    if res is not None and res[1] is not None:
        per_iter_samples_storage[k] = res[1]

for iter_idx in range(START_ITER, N_ITERATIONS):
    print(f'\n{"="*70}')
    print(f'  ITERAZIONE {iter_idx} / {N_ITERATIONS-1}  —  {time.strftime("%H:%M:%S")}')
    print(f'{"="*70}')

    iter_record = {'iter_idx': iter_idx}
    iter_t0 = time.perf_counter()

    # Spostiamo la rete su CPU per il self-play (i worker non usano GPU)
    net_cpu = RisikoNet()
    net_cpu.load_state_dict({k: v.cpu() for k, v in net.state_dict().items()})
    net_cpu.eval()

    # ─────────────── 1. Self-play ───────────────
    print(f'\n[1/5] Self-play {N_GAMES} partite parallele (n_sim={N_SIM}, {N_WORKER} worker)...')
    t0 = time.perf_counter()
    samples_pp, stats_pp = gioca_n_partite_parallele(
        net=net_cpu, n_partite=N_GAMES, n_worker=N_WORKER,
        base_seed=10000 + iter_idx*1000,
        n_simulations=N_SIM, max_decisioni=MAX_DEC,
    )
    wall_sp = time.perf_counter() - t0
    n_vinte = sum(1 for s in stats_pp if s['vincitore'] is not None)
    n_blu_w = sum(1 for s in stats_pp if s['vincitore'] == 'BLU')
    n_rosso_w = sum(1 for s in stats_pp if s['vincitore'] == 'ROSSO')
    tot_samples = sum(len(s) for s in samples_pp)
    print(f'  Self-play OK in {wall_sp/60:.1f} min: {n_vinte}/{N_GAMES} con vincitore '
          f'(BLU={n_blu_w}, ROSSO={n_rosso_w}), {tot_samples} sample')
    iter_record.update(
        n_vinte=n_vinte, n_blu_w=n_blu_w, n_rosso_w=n_rosso_w,
        tot_samples=tot_samples, wall_sp_min=wall_sp/60,
    )

    # ─────────────── 2. Aggiorna buffer ───────────────
    print(f'\n[2/5] Aggiorno buffer cumulativo (sliding window {BUFFER_WINDOW} iter)...')
    per_iter_samples_storage[iter_idx] = samples_pp
    # Drop iterazioni fuori finestra
    drop_idx = iter_idx - BUFFER_WINDOW
    if drop_idx in per_iter_samples_storage and drop_idx >= 0:
        del per_iter_samples_storage[drop_idx]
    # Ricostruisci buffer da zero per evitare drift
    buffer = ReplayBuffer(max_size=500_000, seed=0)
    iter_keys = sorted(per_iter_samples_storage.keys())
    for k in iter_keys:
        for partita_samples in per_iter_samples_storage[k]:
            buffer.add_partita(partita_samples)
    print(f'  Buffer: {len(buffer)} sample da iter {iter_keys}')
    iter_record.update(buffer_size=len(buffer), buffer_iters=iter_keys)

    # ─────────────── 3. Training ───────────────
    print(f'\n[3/5] Training {N_TRAIN_STEPS} step (batch={BATCH_SIZE}) su {DEVICE}...')
    trainer = Trainer(net.to(DEVICE), lr=LR, device=DEVICE)
    losses_log = {'value': [], 'policy': [], 'total': [], 'grad': []}
    t0 = time.perf_counter()
    for step in range(N_TRAIN_STEPS):
        batch = buffer.sample(BATCH_SIZE)
        out = trainer.train_step(batch)
        for k_short, k_full in [('value','value_loss'),('policy','policy_loss'),
                                  ('total','total_loss'),('grad','grad_norm')]:
            losses_log[k_short].append(out[k_full])
    wall_tr = time.perf_counter() - t0
    head_v = float(np.mean(losses_log['value'][:50]))
    tail_v = float(np.mean(losses_log['value'][-50:]))
    head_p = float(np.mean(losses_log['policy'][:50]))
    tail_p = float(np.mean(losses_log['policy'][-50:]))
    print(f'  Training OK in {wall_tr:.0f}s')
    print(f'    value_loss: {head_v:.3f} -> {tail_v:.3f} ({(tail_v-head_v)/max(head_v,1e-6)*100:+.0f}%)')
    print(f'    policy_loss: {head_p:.3f} -> {tail_p:.3f} ({(tail_p-head_p)/max(head_p,1e-6)*100:+.0f}%)')
    iter_record.update(
        value_loss_head=head_v, value_loss_tail=tail_v,
        policy_loss_head=head_p, policy_loss_tail=tail_p,
        wall_tr_s=wall_tr,
    )

    # ─────────────── 4. Eval rete_k vs rete_k-1 ───────────────
    print(f'\n[4/5] Eval inter-rete: nuova vs precedente...')
    # Carica la rete precedente
    prev_iter = iter_idx - 1
    res_prev = carica_iter_checkpoint(prev_iter, RUN_DIR)
    if res_prev is None:
        # Per iter 0 confrontiamo con la rete iniziale (-1)
        net_old = RisikoNet()
        net_old.load_state_dict(
            torch.load(f'{RUN_DIR}/iter_-1/net.pt', map_location='cpu', weights_only=True)
        )
        net_old.eval()
        prev_label = 'iniziale'
    else:
        net_old, _ = res_prev
        prev_label = f'iter_{prev_iter}'

    # Sposta la rete corrente su CPU per match (worker su CPU)
    net_for_match = RisikoNet()
    net_for_match.load_state_dict({k: v.cpu() for k, v in net.state_dict().items()})
    net_for_match.eval()

    t0 = time.perf_counter()
    wr, ci, detail = eval_inter_rete(
        net_new=net_for_match, net_old=net_old,
        n_per_color=EVAL_GAMES_PER_COLOR, n_worker=N_WORKER,
        base_seed=20000 + iter_idx*100,
        n_sim=EVAL_N_SIM, max_dec=EVAL_MAX_DEC, temp=EVAL_TEMP,
    )
    wall_ev = time.perf_counter() - t0
    print(f'  Eval OK in {wall_ev/60:.1f} min')
    print(f'    rete iter_{iter_idx} vs {prev_label}: {wr:.1%} CI95 [{ci[0]:.1%}, {ci[1]:.1%}]')
    print(f'    dettaglio: nuova vince {detail["wins_new_as_blu"]}/{EVAL_GAMES_PER_COLOR} come BLU, '
          f'{detail["wins_new_as_rosso"]}/{EVAL_GAMES_PER_COLOR} come ROSSO')
    iter_record.update(
        eval_vs=prev_label,
        eval_win_rate=wr, eval_ci=list(ci),
        eval_detail=detail,
        wall_ev_min=wall_ev/60,
    )

    # ─────────────── 5. Salva checkpoint ───────────────
    print(f'\n[5/5] Salvo checkpoint iter {iter_idx}...')
    iter_record['wall_total_min'] = (time.perf_counter() - iter_t0) / 60
    iter_record['losses_log'] = losses_log
    history_log['iterations'].append(iter_record)
    salva_iter_checkpoint(iter_idx, net_for_match, samples_pp, history_log, RUN_DIR)
    print(f'  Iter {iter_idx} totale: {iter_record["wall_total_min"]:.0f} min')

print(f'\n{"="*70}\n  TUTTE LE {N_ITERATIONS} ITERAZIONI COMPLETATE\n{"="*70}')

## 5. Riepilogo + plot

In [ ]:
import matplotlib.pyplot as plt

# Tabella riassuntiva
print(f'\n{"iter":>4} | {"vinte":>5} | {"buffer":>7} | {"value_loss":>11} | {"win rate vs prev":>17}')
print('-'*72)
for ir in history_log['iterations']:
    vl = f'{ir["value_loss_head"]:.2f}->{ir["value_loss_tail"]:.2f}'
    wr_str = f'{ir["eval_win_rate"]:.1%}'
    print(f'{ir["iter_idx"]:>4} | {ir["n_vinte"]:>5} | {ir["buffer_size"]:>7} | {vl:>11} | {wr_str:>17}'
          f'  vs {ir["eval_vs"]}')

# Plot win rate per iter
iters = [ir['iter_idx'] for ir in history_log['iterations']]
wrs = [ir['eval_win_rate'] for ir in history_log['iterations']]
ci_los = [ir['eval_ci'][0] for ir in history_log['iterations']]
ci_his = [ir['eval_ci'][1] for ir in history_log['iterations']]

fig, ax = plt.subplots(figsize=(8,4))
ax.fill_between(iters, ci_los, ci_his, alpha=0.2, color='steelblue')
ax.plot(iters, wrs, 'o-', color='steelblue', lw=2, ms=7)
ax.axhline(0.5, ls='--', color='gray', alpha=0.5, label='50% (= no miglioramento)')
ax.set_xlabel('iterazione'); ax.set_ylabel('win rate vs precedente')
ax.set_title('Eval generazionale: rete_N vs rete_N-1')
ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 6. Eval finale vs random (sanity check)

Solo dopo aver completato le 5 iterazioni. Misura la rete finale contro il bot random interno. **Atteso debole** (vedi analisi del notebook diagnostico: rete + MCTS argmax soffre la 'concentrazione su un'azione'). Quando avremo Dirichlet noise (PR4), questo numero salirà significativamente.

In [ ]:
# Carica la rete finale
net_final, _ = carica_iter_checkpoint(N_ITERATIONS - 1, RUN_DIR)

print(f'Eval finale vs random — 40 partite, temp={EVAL_TEMP}...')
t0 = time.perf_counter()
stats_final = gioca_n_partite_vs_random_parallele(
    net=net_final, n_partite=40, n_worker=N_WORKER,
    base_seed=99000, bot_color='BLU',
    n_simulations=EVAL_N_SIM, max_decisioni=EVAL_MAX_DEC,
    temperature_drop_step=0,  # argmax con T=0 default — qui passiamo step=0
)
wall = time.perf_counter() - t0

n_blu_w = sum(1 for s in stats_final if s['vincitore'] == 'BLU')
n_rosso_w = sum(1 for s in stats_final if s['vincitore'] == 'ROSSO')
wr = n_blu_w / 40
ci = wilson_ci(n_blu_w, 40)
print(f'  Eval OK in {wall/60:.1f} min')
print(f'  rete iter {N_ITERATIONS-1} vs random: {wr:.1%} CI95 [{ci[0]:.1%}, {ci[1]:.1%}]')
print(f'  dettaglio: BLU vince {n_blu_w}, ROSSO {n_rosso_w}')

# Salva
with open(f'{RUN_DIR}/eval_final_vs_random.json', 'w') as f:
    json.dump({
        'win_rate': wr, 'ci': list(ci),
        'wins_blu': n_blu_w, 'wins_rosso': n_rosso_w,
        'config': {'temp': EVAL_TEMP, 'n_sim': EVAL_N_SIM, 'n_partite': 40},
    }, f, indent=2)
print(f'\nRisultato salvato in {RUN_DIR}/eval_final_vs_random.json')

## Resume da iter K (in caso di disconnessione Colab)

Se Colab si scollega a metà run:

1. Riapri il notebook su un nuovo runtime
2. Esegui le celle 0, 1
3. **Nella cella di Mount Drive**: cambia `RUN_TIMESTAMP = 'new'` con il timestamp della run che vuoi riprendere (es. `'20260506_0930'`). La cartella esiste già su Drive
4. **Nella cella Knobs**: imposta `START_ITER = K` dove K è l'iter da cui riprendere (es. se hai completato 0,1,2 e vuoi fare 3,4 → `START_ITER = 3`)
5. Esegui tutte le celle in ordine. Il loop ricostruisce automaticamente buffer e history.